## ColdCode: Intelligent Code-Assisted Programming Expert for Beginners

This version has migrated core logic such as Prompt, long code analysis, model invocation, result extraction, and file streams to `coldcode_core`. The notebook only retains the JupyterLab front-end shell.

In [ ]:
import ipywidgets as W
from IPython.display import display, HTML as DHTML
import httpx, json, re, time, hashlib, datetime, os, shutil
from pathlib import Path
import time

In [ ]:
# ===== ColdCode Modular core import =====

import sys
from pathlib import Path

_SEARCH_ROOTS = [
    Path.cwd(),
    Path.cwd() / "coldcode_modular",
    Path.cwd().parent,
]
for _root in _SEARCH_ROOTS:
    if (_root / "src").exists():
        _root_str = str(_root)
        if _root_str not in sys.path:
            sys.path.insert(0, _root_str)

from src import (
    MODEL_FAST,
    MODEL_STRONG,
    MODE_HELP,
    LAST_OUTPUT,
    normalize_file_path,
    load_text_file,
    apply_fixed_code_to_file,
    restore_backup_file,
    build_prompt_compare_text,
    export_markdown_result,
    export_tech_report,
    run_task_stream,
)


In [ ]:
# Core logic such as Prompt, long code analysis, model invocation, and extractor has been migrated to coldcode_core.

# This notebook now only retains the JupyterLab front-end shell and button events.


In [ ]:
# ===== UI Design Section =====

# ===================== Global CSS Enhancement =====================
display(DHTML("""
<style>
/* Overall impression */
.cc-card {
    background: #ffffff;
    border: 1px solid #e2e8f0;
    border-radius: 18px;
    padding: 16px 18px;
    box-shadow: 0 6px 18px rgba(15, 23, 42, 0.06);
    margin-bottom: 14px;
}

.cc-header-card {
    background: linear-gradient(135deg, #f8fbff 0%, #eef6ff 100%);
    border: 1px solid #dbeafe;
    border-radius: 20px;
    padding: 20px 24px;
    box-shadow: 0 8px 20px rgba(59, 130, 246, 0.08);
    margin-bottom: 14px;
}

.cc-section-title {
    font-size: 15px;
    font-weight: 700;
    color: #334155;
    margin: 2px 0 12px 0;
    letter-spacing: 0.2px;
}

.cc-field-label {
    font-size: 14px;
    font-weight: 700;
    color: #334155;
    margin: 4px 0 6px 2px;
}

.cc-subtext {
    font-size: 13px;
    color: #64748b;
    margin-top: 2px;
}

.cc-status-chip {
    display: inline-block;
    padding: 6px 12px;
    border-radius: 999px;
    font-size: 13px;
    font-weight: 700;
    background: #eff6ff;
    color: #1d4ed8;
    border: 1px solid #bfdbfe;
}

.cc-tip-box {
    background: #f8fafc;
    border: 1px solid #e2e8f0;
    border-radius: 14px;
    padding: 10px 12px;
}

.cc-kbd {
    display: inline-block;
    padding: 1px 6px;
    border-radius: 8px;
    border: 1px solid #cbd5e1;
    background: #f8fafc;
    font-family: monospace;
    font-size: 12px;
}

/* General widget polishing */
.jupyter-widgets .widget-button {
    border-radius: 12px !important;
    font-weight: 700 !important;
    box-shadow: none !important;
}

.jupyter-widgets .widget-text input,
.jupyter-widgets .widget-textarea textarea,
.jupyter-widgets .widget-dropdown select {
    border-radius: 12px !important;
    border: 1px solid #cbd5e1 !important;
    box-shadow: none !important;
    padding-left: 8px !important;
}

.jupyter-widgets .widget-text input:focus,
.jupyter-widgets .widget-textarea textarea:focus,
.jupyter-widgets .widget-dropdown select:focus {
    border-color: #60a5fa !important;
    box-shadow: 0 0 0 3px rgba(96, 165, 250, 0.15) !important;
}

/* Output Area */
.cc-output-box {
    background: #0f172a;
    color: #e2e8f0 !important;
    border: 1px solid #1e293b;
    border-radius: 16px;
    padding: 10px 12px;
}

.cc-output-box * {
    color: #e2e8f0 !important;
}

.cc-output-box pre,
.cc-output-box code,
.cc-output-box .output,
.cc-output-box .output_text,
.cc-output-box .output_subarea,
.cc-output-box .jp-OutputArea-output,
.cc-output-box .jp-RenderedText,
.cc-output-box .jp-OutputArea-child {
    color: #e2e8f0 !important;
    background: transparent !important;
}

/* Makes VBox/HBox cards look more natural */
.cc-no-bottom {
    margin-bottom: 0 !important;
}
</style>
"""))


# ==================== Small utility functions =====================
def section_title(text):
    return W.HTML(f"<div class='cc-section-title'>{text}</div>")

def field_label(text):
    return W.HTML(f"<div class='cc-field-label'>{text}</div>")

# ===================== original control (retain variable names) ====================

# title
title = W.HTML(
    value="""
    <div style="text-align:center;">
        <div style="
            font-size:38px;
            font-weight:800;
            color:#1e293b;
            line-height:1.2;
            margin-bottom:6px;
            letter-spacing:0.3px;
        ">
            💦 Coldrain's ColdCode
        </div>
        <div style="
            font-size:15px;
            color:#64748b;
            margin-bottom:12px;
        ">
            Debug · Explain · Refactor · Scaffold/Test · ROCm Doctor
        </div>
        <div style="
            height:3px;
            border-radius:999px;
            background:linear-gradient(to right, #38bdf8, #60a5fa, #818cf8);
            width:100%;
        "></div>
    </div>
    """
)

# mode Select
mode_dd = W.Dropdown(
    options=["Debug", "Explain", "Refactor", "Scaffold/Test", "ROCm Doctor"],
    value="Debug",
    description="🚀Mode:",
    layout=W.Layout(width='240px', margin='4px 10px 4px 0')
)

# language Select
lang_dd = W.Dropdown(
    options=["Python", "Java", "C++"],
    value="Python",
    description="📖Language:",
    layout=W.Layout(width='220px', margin='4px 10px 4px 0')
)

# model Select
model_dd = W.Dropdown(
    options=[("Fast llama3.1:8b", MODEL_FAST), ("High-quality qwen3-coder:30b", MODEL_STRONG)],
    value=MODEL_FAST,
    description="🤖Model:",
    layout=W.Layout(width='260px', margin='4px 10px 4px 0')
)

# prompt version select
prompt_ver_dd = W.Dropdown(
    options=[("v0 Normal", "v0"), ("v1 Structured", "v1"), ("v2 few-shot", "v2"), ("v3 Engineering", "v3")],
    value="v3",
    description="🚄Prompt:",
    layout=W.Layout(width='220px', margin='4px 10px 4px 0')
)

# File Workflow Control
file_path_in = W.Text(
    placeholder="For example: src/main.py or ./demo.py",
    description="📁File Path:",
    layout=W.Layout(width="100%", height='40px')
)

load_file_btn = W.Button(
    description="Load File",
    button_style="",
    icon="folder-open",
    layout=W.Layout(width='130px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Load file content into the code editor"
)

apply_file_btn = W.Button(
    description="Apply to File",
    button_style="info",
    icon="save",
    layout=W.Layout(width='140px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Write the fixed code back to the file"
)

restore_file_btn = W.Button(
    description="Restore Backup",
    button_style="",
    icon="history",
    layout=W.Layout(width='150px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Restore file from .bak backup"
)

file_info = W.HTML(
    value="<div class='cc-subtext'>Current file not loaded. You can also paste code directly into the text area below.</div>",
    layout=W.Layout(width='100%', margin='2px 0 8px 0')
)

# Parameter fine-tuning box
num_predict_sl = W.IntSlider(
    value=350, min=120, max=2000, step=10,
    description="Output Limit:",
    continuous_update=False,
    layout=W.Layout(width='360px', margin='4px 16px 4px 0')
)

temp_sl = W.FloatSlider(
    value=0.2, min=0.0, max=0.8, step=0.05,
    description="temperature:",
    continuous_update=False,
    layout=W.Layout(width='380px', margin='4px 0 4px 0')
)

# input boxes
question_in = W.Text(
    placeholder=MODE_HELP["Debug"]["question_placeholder"],
    description="",
    layout=W.Layout(width="100%", height='40px')
)

code_in = W.Textarea(
    placeholder=MODE_HELP["Debug"]["code_placeholder"],
    description="",
    layout=W.Layout(width="100%", height="240px")
)

tb_in = W.Textarea(
    placeholder=MODE_HELP["Debug"]["tb_placeholder"],
    description="",
    layout=W.Layout(width="100%", height="160px")
)

mode_tip = W.HTML(
    value="",
    layout=W.Layout(width='100%', margin='0 0 6px 0')
)

learning_card_ck = W.Checkbox(
    value=True,
    description="Add error growth cards after Debug (only effective in Debug mode)",
    indent=False,
    layout=W.Layout(width='350px', margin='4px 16px 4px 0')
)

# buttons
run_btn = W.Button(
    description="Run",
    button_style="warning",
    icon="play",
    layout=W.Layout(width='120px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Run current request"
)

apply_btn = W.Button(
    description="Apply Fixed Code",
    button_style="info",
    icon="check",
    layout=W.Layout(width='160px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Apply the fixed code"
)

undo_btn = W.Button(
    description="Undo",
    button_style="",
    icon="rotate-left",
    layout=W.Layout(width='110px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Undo last application"
)

clear_btn = W.Button(
    description="Clear",
    button_style="",
    icon="trash",
    layout=W.Layout(width='110px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Clear inputs and outputs"
)

export_btn = W.Button(
    description="Export .md",
    button_style="success",
    icon="download",
    layout=W.Layout(width='130px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Export Markdown"
)

report_btn = W.Button(
    description="Export Report",
    button_style="primary",
    icon="file-text",
    layout=W.Layout(width='140px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Export Report"
)

compare_btn = W.Button(
    description="Prompt Compare",
    button_style="",
    icon="columns",
    layout=W.Layout(width='160px', height='40px', margin='4px 8px 4px 0'),
    tooltip="Compare the differences between the current mode and different Prompt versions."
)

demo_btn = W.Button(
    description="Fill Demo",
    button_style="",
    icon="magic",
    layout=W.Layout(width='130px', height='40px', margin='4px 0 4px 0'),
    tooltip="Automatically fill the demo inputs for the current mode"
)

apply_btn.disabled = True
apply_file_btn.disabled = True
restore_file_btn.disabled = True
undo_btn.disabled = True

status = W.HTML(
    value="<span class='cc-status-chip'>Ready</span>",
    layout=W.Layout(width="100%", margin='0 0 10px 0')
)

out = W.Output(
    layout=W.Layout(
        width='100%',
        height='460px',
        overflow='auto',
        border='none'
    )
)

prompt_lab_out = W.Output(
    layout=W.Layout(
        width='100%',
        height='220px',
        overflow='auto',
        border='none'
    )
)

# ===================== Control description: Uniform width =====================
for w in [lang_dd, mode_dd, model_dd, prompt_ver_dd, file_path_in]:
    w.style.description_width = '72px'

for w in [num_predict_sl, temp_sl]:
    w.style.description_width = '88px'

# ===================== Top Header =====================
header_box = W.VBox([title], layout=W.Layout(width='100%'))
header_box.add_class("cc-header-card")

# ===================== Control Area =====================
selectbar_row = W.HBox(
    [lang_dd, mode_dd, model_dd, prompt_ver_dd],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

btn_row = W.HBox(
    [run_btn, apply_btn, apply_file_btn, undo_btn, restore_file_btn, clear_btn, export_btn, report_btn],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

advanced_row = W.HBox(
    [learning_card_ck, compare_btn, demo_btn],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

param_row = W.HBox(
    [num_predict_sl, temp_sl],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='center',
        justify_content='flex-start'
    )
)

control_panel = W.VBox(
    [
        section_title("⚙️ Control Panel"),
        selectbar_row,
        section_title("🔧 Features"),
        btn_row,
        W.HTML("<div style='height:8px;'></div>"),
        section_title("✨ Highlights"),
        advanced_row,
        W.HTML("<div style='height:8px;'></div>"),
        section_title("🎛️ Parameters"),
        param_row,
    ],
    layout=W.Layout(width='100%')
)
control_panel.add_class("cc-card")

# ===================== Left Input Area =====================
left_panel = W.VBox(
    [
        section_title("📝 Input Area"),
        mode_tip,
        field_label("📁 Working File"),
        file_path_in,
        file_info,
        W.HBox([load_file_btn], layout=W.Layout(width='100%', justify_content='flex-start')),
        W.HTML("<div style='height:8px;'></div>"),
        field_label("❓ Problem Description"),
        question_in,
        W.HTML("<div style='height:8px;'></div>"),
        field_label("💻 Code Content / Installation Commands"),
        code_in,
        W.HTML("<div style='height:8px;'></div>"),
        field_label("⚠️ Error / Log / Environment Information"),
        tb_in,
    ],
    layout=W.Layout(
        flex='1 1 720px',
        width='50%',
        min_width='680px'
    )
)
left_panel.add_class("cc-card")

# ===================== Right Output Area =====================
output_title = W.HTML("""
<div class='cc-section-title'>
    📦 Output Area
    <div class='cc-subtext'>Model response, repair suggestions, and patch results will all be displayed here.</div>
</div>
""")

output_wrap = W.VBox([out], layout=W.Layout(width='100%'))
output_wrap.add_class("cc-output-box")

prompt_lab_title = W.HTML("""
<div class='cc-section-title'>
    🧪 Prompt Lab
    <div class='cc-subtext'>Click Prompt Compare to view the differences in prompts under the current mode across v0-v3.</div>
</div>
""")

prompt_lab_wrap = W.VBox([prompt_lab_out], layout=W.Layout(width='100%'))
prompt_lab_wrap.add_class("cc-output-box")

right_panel = W.VBox(
    [
        output_title,
        status,
        output_wrap,
        W.HTML("<div style='height:12px;'></div>"),
        prompt_lab_title,
        prompt_lab_wrap
    ],
    layout=W.Layout(
        flex='1 1 720px',
        width='50%',
        min_width='680px'
    )
)
right_panel.add_class("cc-card")

# ===================== Main Workspace =====================
workspace = W.HBox(
    [left_panel, right_panel],
    layout=W.Layout(
        width='100%',
        flex_flow='row wrap',
        align_items='stretch',
        justify_content='space-between'
    )
)


In [ ]:
def on_clear(_):
    """
    Clear the output area without clearing the input content.
    """
    out.clear_output()
    status.value = "<span class='cc-status-chip'>Ready</span>"
    with prompt_lab_out:
        prompt_lab_out.clear_output()
        print("Click Prompt Compare to view the differences in prompts under the current mode across v0-v3.")


def update_file_info(path: str = "", *, message: str = "", ok: bool = True):
    if path:
        safe = path.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        color = "#0f766e" if ok else "#b45309"
        extra = f"<div class='cc-subtext' style='margin-top:4px;color:{color};'>{message}</div>" if message else ""
        file_info.value = f"<div class='cc-subtext'><b>Current File:</b><code>{safe}</code></div>{extra}"
    else:
        file_info.value = "<div class='cc-subtext'>No file loaded. You can also paste code directly into the text area below.</div>"


def render_mode_tip(mode: str):
    help_pack = MODE_HELP[mode]
    badge = "🩺 ROCm Doctor" if mode == "ROCm Doctor" else f"✨ {mode} Mode"
    mode_tip.value = (
        "<div class='cc-tip-box'>"
        f"<div style='font-weight:700;color:#0f172a;margin-bottom:4px;'>{badge}</div>"
        f"<div class='cc-subtext' style='margin-top:0;color:#475569;'>{help_pack['hint']}</div>"
        "</div>"
    )


def sync_placeholders(mode: str):
    help_pack = MODE_HELP[mode]
    question_in.placeholder = help_pack["question_placeholder"]
    code_in.placeholder = help_pack["code_placeholder"]
    tb_in.placeholder = help_pack["tb_placeholder"]


def on_mode_change(change=None):
    mode = mode_dd.value
    render_mode_tip(mode)
    sync_placeholders(mode)
    learning_card_ck.disabled = (mode != "Debug")
    if mode != "Debug":
        learning_card_ck.tooltip = "This switch is only effective in Debug mode"
    else:
        learning_card_ck.tooltip = "When enabled, error growth cards will be appended to the end of the debug results"


def on_compare_prompt(_):
    text = build_prompt_compare_text(mode_dd.value)
    with prompt_lab_out:
        prompt_lab_out.clear_output()
        print(text)


def on_fill_demo(_):
    mode = mode_dd.value
    if mode == "Debug":
        lang_dd.value = "Python"
        question_in.value = "Please help me identify the issue and provide the minimal fix steps, diff, and the corrected code."
        code_in.value = "def add(a, b):\n    return a + b\n\nprint(ad(1, 2))\n"
        tb_in.value = (
            'Traceback (most recent call last):\n'
            '  File "demo.py", line 4, in <module>\n'
            '    print(ad(1, 2))\n'
            "NameError: name 'ad' is not defined\n"
        )
        status.value = "<b style='color:green'>Demo example filled.</b>"
        return

    if mode == "Explain":
        lang_dd.value = "Python"
        question_in.value = "Please explain this code in a high-level overview, line by line, and highlight key concepts."
        code_in.value = (
            "def moving_average(nums, window=3):\n"
            "    if window <= 0:\n"
            "        raise ValueError('window must be positive')\n"
            "    result = []\n"
            "    for i in range(len(nums)):\n"
            "        left = max(0, i - window + 1)\n"
            "        chunk = nums[left:i+1]\n"
            "        result.append(sum(chunk) / len(chunk))\n"
            "    return result\n"
        )
        tb_in.value = ""
        status.value = "<b style='color:green'>Demo example filled.</b>"
        return

    if mode == "Refactor":
        lang_dd.value = "Python"
        question_in.value = "Please make the minimal refactor to improve readability without changing the functionality."
        code_in.value = (
            "def score(data):\n"
            "    s = 0\n"
            "    for i in range(len(data)):\n"
            "        if data[i] > 60:\n"
            "            s = s + data[i]\n"
            "    return s\n"
        )
        tb_in.value = ""
        status.value = "<b style='color:green'>Demo example filled.</b>"
        return

    if mode == "Scaffold/Test":
        lang_dd.value = "Python"
        question_in.value = "Please generate a command-line Todo project with pytest tests and run instructions."
        code_in.value = ""
        tb_in.value = "Requirements: Supports the add, list, and done commands; clear directory structure; minimal and executable size."
        status.value = "<b style='color:green'>Demo example filled.</b>"
        return

    if mode == "ROCm Doctor":
        lang_dd.value = "Python"
        question_in.value = "I have installed ROCm and PyTorch on Ubuntu, rocminfo can see the GPU, but torch.cuda.is_available() returns False. Please help me identify the most likely issue and provide minimal troubleshooting steps."
        code_in.value = (
            "python -c \"import torch; "
            "print(torch.__version__); "
            "print(torch.version.hip); "
            "print(torch.cuda.is_available())\""
        )
        tb_in.value = (
            "$ rocminfo | head -n 5\n"
            "ROCk module is loaded\n"
            "=====================\n"
            "HSA Agents:\n"
            "  Agent 1: gfx1100\n\n"
            "$ python -c \"import torch; print(torch.__version__); print(torch.version.hip); print(torch.cuda.is_available())\"\n"
            "2.3.0\n"
            "None\n"
            "False\n\n"
            "$ python -m pip show torch\n"
            "Name: torch\n"
            "Version: 2.3.0\n"
            "Summary: Tensors and Dynamic neural networks in Python\n"
        )
        status.value = "<b style='color:green'>Demo example filled.</b>"
        return


def on_load_file(_):
    try:
        info = load_text_file(file_path_in.value)
        code_in.value = info["content"]
        file_path_in.value = info["path"]
        LAST_OUTPUT["loaded_file_path"] = info["path"]
        LAST_OUTPUT["backup_file_path"] = info.get("backup_path", "")
        apply_file_btn.disabled = True
        restore_file_btn.disabled = not info.get("backup_exists", False)
        update_file_info(info["path"], message="Demo example filled.", ok=True)
        status.value = f"<b style='color:green'>Demo example filled.</b> <code>{info['path']}</code>"
    except Exception as e:
        update_file_info("", message="Demo example filled.", ok=False)
        apply_file_btn.disabled = True
        restore_file_btn.disabled = True
        status.value = f"<b style='color:#b00'>Failed to load file.</b> {e}"


def on_apply_file(_):
    fixed = LAST_OUTPUT.get("fixed_code", "").strip()
    if not fixed:
        status.value = "<b style='color:#b00'>Failed to detect fixed code. Please run and ensure the output contains the fixed code.</b>"
        return

    path = normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", ""))
    if not path:
        status.value = "<b style='color:#b00'>Please input and load the target file first, then execute Apply to File.</b>"
        return

    try:
        info = apply_fixed_code_to_file(path, fixed)
        code_in.value = info["content"]
        file_path_in.value = info["path"]
        LAST_OUTPUT["loaded_file_path"] = info["path"]
        LAST_OUTPUT["backup_file_path"] = info["backup_path"]
        LAST_OUTPUT["last_apply_target"] = "file"
        restore_file_btn.disabled = False
        apply_file_btn.disabled = False
        update_file_info(info["path"], message=f"Failed to detect fixed code. Please run and ensure the output contains the fixed code.", ok=True)
        status.value = f"<b style='color:green'>Failed to detect fixed code. Please run and ensure the output contains the fixed code.</b> <code>{info['path']}</code>（备份：<code>{Path(info['backup_path']).name}</code>）"
    except Exception as e:
        status.value = f"<b style='color:#b00'>Failed to apply fixed code to file.</b> {e}"


def on_restore_backup(_):
    path = normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", ""))
    if not path:
        status.value = "<b style='color:#b00'>Please specify the file path first.</b>"
        return

    try:
        info = restore_backup_file(path)
        code_in.value = info["content"]
        file_path_in.value = info["path"]
        LAST_OUTPUT["loaded_file_path"] = info["path"]
        LAST_OUTPUT["backup_file_path"] = info["backup_path"]
        LAST_OUTPUT["last_apply_target"] = "restore_backup"
        restore_file_btn.disabled = False
        update_file_info(info["path"], message=f"Failed to detect fixed code. Please run and ensure the output contains the fixed code.", ok=True)
        status.value = f"<b style='color:green'>Failed to detect fixed code. Please run and ensure the output contains the fixed code.</b> <code>{info['path']}</code>"
    except Exception as e:
        status.value = f"<b style='color:#b00'>Failed to restore backup.</b> {e}"


def on_export(_):
    try:
        result_file, log_file = export_markdown_result('.')
        status.value = f"<b style='color:green'>Export successful:</b> <code>{result_file.name}</code> and <code>{log_file.name}</code> (current directory)"
    except Exception as e:
        status.value = f"<b style='color:#b00'>Export failed:</b> {e}"


def on_export_report(_):
    try:
        report_file = export_tech_report('.')
        status.value = f"<b style='color:green'>Export successful:</b> <code>{report_file.name}</code>"
    except Exception as e:
        status.value = f"<b style='color:#b00'>Export failed:</b> {e}"


def on_apply(_):
    fixed = LAST_OUTPUT.get("fixed_code", "").strip()
    if not fixed:
        status.value = "<b style='color:#b00'>Failed to detect fixed code. Please run and ensure the output contains the fixed code.</b>"
        return

    current = code_in.value or ""
    if not current.strip():
        status.value = "<b style='color:#b00'>Code box is empty, but you can still apply the fix. It is recommended to put the original code in for comparison.</b>"

    LAST_OUTPUT["prev_code"] = current
    code_in.value = fixed + ("\n" if not fixed.endswith("\n") else "")
    undo_btn.disabled = False
    apply_file_btn.disabled = (normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", "")) == "")
    LAST_OUTPUT["last_apply_target"] = "code_box"
    status.value = "<b style='color:green'>Successfully applied fixed code to code box ✅</b>（To rewrite the original file, please click: Apply to File）"


def on_undo(_):
    prev = LAST_OUTPUT.get("prev_code", "")
    if not prev:
        status.value = "<b style='color:#b00'>No editable history.</b>"
        return
    code_in.value = prev
    LAST_OUTPUT["prev_code"] = ""
    undo_btn.disabled = True
    status.value = "<b style='color:green'>Successfully reverted code box modification ✅</b>"


def on_run(_):
    run_btn.disabled = True
    apply_btn.disabled = True
    out.clear_output()
    status.value = "<span class='cc-status-chip'>Running...</span>"

    active_file_path = normalize_file_path(file_path_in.value or LAST_OUTPUT.get("loaded_file_path", ""))

    try:
        with out:
            result = run_task_stream(
                lang=lang_dd.value,
                mode=mode_dd.value,
                model=model_dd.value,
                prompt_ver=prompt_ver_dd.value,
                code=code_in.value or "",
                tb=tb_in.value or "",
                question=question_in.value or "",
                file_path=active_file_path,
                num_predict=int(num_predict_sl.value),
                temperature=float(temp_sl.value),
                learning_card=bool(mode_dd.value == "Debug" and learning_card_ck.value),
                output_callback=lambda delta: print(delta, end="", flush=True),
            )

        fixed = result.get("fixed_code", "").strip()
        apply_btn.disabled = (fixed == "")
        apply_file_btn.disabled = (fixed == "" or not active_file_path)

        if active_file_path:
            LAST_OUTPUT["loaded_file_path"] = active_file_path
            restore_file_btn.disabled = not Path(active_file_path + ".bak").exists()
            update_file_info(active_file_path, message="Current result generated based on loaded file.", ok=True)

        if result.get("cache_hit"):
            status.value = "<b style='color:green'>Finished ✅</b>"
        elif fixed:
            if active_file_path:
                status.value = "<b style='color:green'>Finished ✅</b>（The repaired code has been detected. You can click Apply or Apply to File.）"
            else:
                status.value = "<b style='color:green'>Finished ✅</b>（The repaired code has been detected. You can click Apply.）"
        else:
            status.value = "<b style='color:green'>Finished ✅</b>（No fixed code detected.）"
    except Exception as e:
        status.value = f"<b style='color:#b00'>Failed:</b> {e}"
    finally:
        run_btn.disabled = False


In [ ]:
# Bind button events
run_btn.on_click(on_run)
load_file_btn.on_click(on_load_file)
apply_btn.on_click(on_apply)
apply_file_btn.on_click(on_apply_file)
restore_file_btn.on_click(on_restore_backup)
undo_btn.on_click(on_undo)
clear_btn.on_click(on_clear)
export_btn.on_click(on_export)
report_btn.on_click(on_export_report)
compare_btn.on_click(on_compare_prompt)
demo_btn.on_click(on_fill_demo)
mode_dd.observe(on_mode_change, names='value')


In [ ]:
# ===================== Final UI =====================
ui = W.VBox(
    [
        header_box,
        control_panel,
        workspace
    ],
    layout=W.Layout(width='100%')
)

with out:
    print("Welcome to Coldrain's ColdCode")
    print("Enter your question, code, and error messages on the left, then click Run.")
    print("New features: ROCm Doctor, Error Growth Cards, Prompt Compare, Fill Demo.")

with prompt_lab_out:
    print("Click Prompt Compare to view the differences in prompts under the current mode across v0-v3.")
on_mode_change()

display(ui)


Please help me fix this, providing the minimum steps and diff.

def add(a, b):

return a + b

print(ad(1, 2))

Traceback (most recent call last):

File "demo.py", line 4, in <module>

print(ad(1, 2))

NameError: name 'ad' is not defined

---

You can also try the new ROCm Doctor mode. Example problem:

rocminfo can see the GPU, but `python -c "import torch; print(torch.version.hip); print(torch.cuda.is_available())"` outputs `None` and `False`. Where does this seem to be going wrong?